# Campus Hazard Detection — Meta-Classifier Full Pipeline

This notebook contains the complete meta-classifier workflow:

1. Load four trained YOLO models and the global label mapping.
2. Run all models on images in `backend/test_cases/`.
3. Harmonise labels and group overlapping predictions using IoU.
4. Extract meta-classifier features and save `meta_training_dataset.csv`.
5. Train and evaluate an MLP meta-classifier.
6. Save the trained model, metrics, classification report, confusion matrix, and inference files.
7. Run final MLP-based ensemble inference and save annotated output images.

> **Important:** For a strict final evaluation, add a `human_verified_label` column to the CSV and use it as the target. If it does not exist, this notebook uses `target_global_label` as a prototype rule-based pseudo-label.

In [ ]:
# Optional: run only if packages are missing
# !pip -q install ultralytics joblib scikit-learn pillow pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import csv
import shutil
from collections import Counter
from statistics import mean

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

BACKEND_DIR = Path("/content/drive/MyDrive/CSC4602_Group_Project/backend")
TEST_CASES_DIR = BACKEND_DIR / "test_cases"
META_DIR = BACKEND_DIR / "meta_classifier"
SUBMISSION_DIR = BACKEND_DIR / "meta_classifier_submission"
RESULTS_DIR = SUBMISSION_DIR / "results"

META_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Backend:", BACKEND_DIR)
print("Test images folder:", TEST_CASES_DIR)

## 1. Load global classes, mappings, and four YOLO models

In [ ]:
GLOBAL_CLASSES = [
    "pothole",
    "cracked_pavement",
    "wet_slippery_floor",
    "surface_damage",
    "damaged_sidewalk",
    "obstacle_on_walkway",
    "construction_debris",
    "fallen_branch",
    "traffic_cone",
    "open_drain",
    "uncovered_manhole",
    "uneven_pavement",
    "missing_barricade",
    "damaged_warning_sign",
    "broken_handrail",
    "blocked_walkway"
]

GLOBAL_CLASS_INDEX = {label: idx for idx, label in enumerate(GLOBAL_CLASSES)}

mapping_path = BACKEND_DIR / "global_label_mapping.json"
with open(mapping_path, "r", encoding="utf-8") as f:
    global_label_mapping = json.load(f)

models = {
    "member1": YOLO(str(BACKEND_DIR / "member1_best.pt")),
    "member2": YOLO(str(BACKEND_DIR / "member2_best.pt")),
    "member3": YOLO(str(BACKEND_DIR / "member3_best.pt")),
    "member4": YOLO(str(BACKEND_DIR / "member4_best.pt")),
}

print("Loaded 4 YOLO models.")
print("Global classes:", len(GLOBAL_CLASSES))

## 2. Helper functions: detection extraction, IoU, related-label groups

In [ ]:
def extract_detections(result, class_mapping, model_name):
    detections = []
    for box in result.boxes:
        class_id = int(box.cls[0].item())
        confidence = float(box.conf[0].item())
        xyxy = [round(float(x), 2) for x in box.xyxy[0].tolist()]
        detections.append({
            "model": model_name,
            "class_id": class_id,
            "label": class_mapping[class_id],
            "confidence": confidence,
            "box": xyxy
        })
    return detections


def calculate_iou(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    area_a = max(0, box_a[2] - box_a[0]) * max(0, box_a[3] - box_a[1])
    area_b = max(0, box_b[2] - box_b[0]) * max(0, box_b[3] - box_b[1])

    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def related_hazard(label_a, label_b):
    if label_a == label_b:
        return True

    related_groups = [
        {"pothole", "open_drain", "uncovered_manhole"},
        {"surface_damage", "damaged_sidewalk", "uneven_pavement", "cracked_pavement"},
        {"obstacle_on_walkway", "construction_debris", "fallen_branch", "blocked_walkway"},
    ]
    return any(label_a in group and label_b in group for group in related_groups)


def group_related_detections(detections, iou_threshold=0.30):
    used = set()
    grouped = []

    for i, det_a in enumerate(detections):
        if i in used:
            continue

        group = [det_a]
        used.add(i)

        for j, det_b in enumerate(detections):
            if j in used:
                continue

            iou = calculate_iou(det_a["box"], det_b["box"])
            if iou >= iou_threshold and related_hazard(det_a["label"], det_b["label"]):
                group.append(det_b)
                used.add(j)

        grouped.append(group)

    return grouped

## 3. Feature extraction for the meta-classifier

In [ ]:
def build_meta_feature_row(group, image_name, provisional_label):
    confidences = [d["confidence"] for d in group]
    labels = [d["label"] for d in group]
    model_names = sorted(set(d["model"] for d in group))

    best_det = max(group, key=lambda d: d["confidence"])
    x1, y1, x2, y2 = best_det["box"]
    width = max(0, x2 - x1)
    height = max(0, y2 - y1)
    area = width * height

    iou_values = []
    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            iou_values.append(calculate_iou(group[i]["box"], group[j]["box"]))

    label_counts = Counter(labels)

    row = {
        "image": image_name,
        "target_global_label": provisional_label,
        "target_global_id": GLOBAL_CLASS_INDEX[provisional_label],
        "best_confidence": round(max(confidences), 4),
        "mean_confidence": round(mean(confidences), 4),
        "min_confidence": round(min(confidences), 4),
        "agreement_count": len(model_names),
        "detection_count": len(group),
        "max_iou": round(max(iou_values) if iou_values else 0.0, 4),
        "mean_iou": round(mean(iou_values) if iou_values else 0.0, 4),
        "best_box_x1": round(x1, 2),
        "best_box_y1": round(y1, 2),
        "best_box_width": round(width, 2),
        "best_box_height": round(height, 2),
        "best_box_area": round(area, 2),
        "member1_detected": int("member1" in model_names),
        "member2_detected": int("member2" in model_names),
        "member3_detected": int("member3" in model_names),
        "member4_detected": int("member4" in model_names),
    }

    for label in GLOBAL_CLASSES:
        row[f"{label}_votes"] = label_counts[label]

    return row


# Prototype rule used only to create a provisional label when no human label is available.
GROUND_HOLE_PRIORITY = {
    "uncovered_manhole": 3,
    "open_drain": 2,
    "pothole": 1
}

SPECIFICITY_PRIORITY = {
    "uncovered_manhole": 5,
    "open_drain": 5,
    "pothole": 4,
    "damaged_sidewalk": 4,
    "uneven_pavement": 4,
    "cracked_pavement": 4,
    "surface_damage": 2,
    "construction_debris": 4,
    "fallen_branch": 4,
    "obstacle_on_walkway": 3,
    "blocked_walkway": 2,
    "wet_slippery_floor": 4,
    "traffic_cone": 3,
    "missing_barricade": 4,
    "damaged_warning_sign": 4,
    "broken_handrail": 4
}


def choose_provisional_label(group):
    chosen = group[0]

    for candidate in group[1:]:
        a, b = chosen["label"], candidate["label"]

        if a in GROUND_HOLE_PRIORITY and b in GROUND_HOLE_PRIORITY:
            if GROUND_HOLE_PRIORITY[b] > GROUND_HOLE_PRIORITY[a]:
                chosen = candidate
            elif GROUND_HOLE_PRIORITY[b] == GROUND_HOLE_PRIORITY[a]:
                if candidate["confidence"] > chosen["confidence"]:
                    chosen = candidate
        else:
            score_a = SPECIFICITY_PRIORITY.get(a, 1) + chosen["confidence"]
            score_b = SPECIFICITY_PRIORITY.get(b, 1) + candidate["confidence"]
            if score_b > score_a:
                chosen = candidate

    return chosen["label"]

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".webp"}
test_images = sorted([
    p for p in TEST_CASES_DIR.iterdir()
    if p.suffix.lower() in image_extensions
])

print("Found test images:", len(test_images))

meta_rows = []

for image_path in test_images:
    detections = []

    for member_name, model in models.items():
        result = model(str(image_path), conf=0.25, verbose=False)[0]
        mapping = {int(k): v for k, v in global_label_mapping[member_name].items()}
        detections.extend(extract_detections(result, mapping, member_name))

    for group in group_related_detections(detections, iou_threshold=0.30):
        provisional_label = choose_provisional_label(group)
        meta_rows.append(build_meta_feature_row(group, image_path.name, provisional_label))

meta_df = pd.DataFrame(meta_rows)

meta_csv_path = META_DIR / "meta_training_dataset.csv"
meta_df.to_csv(meta_csv_path, index=False)

print("Saved:", meta_csv_path)
print("Rows:", len(meta_df))
print(meta_df["target_global_label"].value_counts())

## 4. Optional human verification

For a strict evaluation, open `meta_training_dataset.csv`, add a `human_verified_label` column, and correct each group manually using the original test image and bounding box. The training cell below automatically prefers that column when available.

## 5. Train and evaluate the MLP meta-classifier

In [ ]:
feature_columns = [
    "best_confidence",
    "mean_confidence",
    "min_confidence",
    "agreement_count",
    "detection_count",
    "max_iou",
    "mean_iou",
    "best_box_x1",
    "best_box_y1",
    "best_box_width",
    "best_box_height",
    "best_box_area",
    "member1_detected",
    "member2_detected",
    "member3_detected",
    "member4_detected"
]

vote_columns = [col for col in meta_df.columns if col.endswith("_votes")]
feature_columns += vote_columns

target_column = (
    "human_verified_label"
    if "human_verified_label" in meta_df.columns and meta_df["human_verified_label"].notna().all()
    else "target_global_label"
)

if target_column == "target_global_label":
    print("WARNING: Using prototype pseudo-labels from rule-based ensemble.")
else:
    print("Using human-verified labels.")

X = meta_df[feature_columns].fillna(0)
y_text = meta_df[target_column].astype(str)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

class_counts = pd.Series(y_text).value_counts()
classes_with_lt2 = class_counts[class_counts < 2]
if len(classes_with_lt2):
    print("Classes with fewer than 2 samples; remove or add samples before stratified split:")
    print(classes_with_lt2)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

meta_classifier = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=0.01,
        learning_rate_init=0.001,
        max_iter=1500,
        random_state=42
    ))
])

meta_classifier.fit(X_train, y_train)
y_pred = meta_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print(f"Accuracy: {accuracy:.3f}")
print(f"Macro Precision: {precision:.3f}")
print(f"Macro Recall: {recall:.3f}")
print(f"Macro F1-score: {f1:.3f}")

In [ ]:
class_labels_present = label_encoder.inverse_transform(sorted(np.unique(np.concatenate([y_test, y_pred]))))

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=sorted(np.unique(np.concatenate([y_test, y_pred])))
)

fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay(cm, display_labels=class_labels_present).plot(
    ax=ax, xticks_rotation=45, colorbar=False
)
plt.title("Meta-Classifier Confusion Matrix")
plt.tight_layout()

cm_path = META_DIR / "meta_classifier_confusion_matrix.png"
plt.savefig(cm_path, dpi=200)
plt.show()

report_text = classification_report(
    y_test,
    y_pred,
    labels=sorted(np.unique(np.concatenate([y_test, y_pred]))),
    target_names=class_labels_present,
    zero_division=0
)

print(report_text)

## 6. Save trained model and required submission files

In [ ]:
joblib.dump(meta_classifier, META_DIR / "meta_classifier_mlp.joblib")
joblib.dump(label_encoder, META_DIR / "label_encoder.joblib")
joblib.dump(feature_columns, META_DIR / "feature_columns.joblib")

# Submission folder
if SUBMISSION_DIR.exists():
    # preserves notebook files if copied later, only refresh required model/results files
    pass

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(META_DIR / "meta_classifier_mlp.joblib", SUBMISSION_DIR / "trained_meta_classifier.pkl")
shutil.copy(META_DIR / "label_encoder.joblib", SUBMISSION_DIR / "label_encoder.joblib")
shutil.copy(META_DIR / "feature_columns.joblib", SUBMISSION_DIR / "feature_columns.joblib")
shutil.copy(cm_path, RESULTS_DIR / "confusion_matrix.png")

with open(RESULTS_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write("Meta-Classifier Classification Report\n")
    f.write("=" * 45 + "\n\n")
    f.write(report_text)

metrics = {
    "dataset_size": int(len(meta_df)),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "num_classes": int(len(label_encoder.classes_)),
    "classes": list(label_encoder.classes_),
    "accuracy": round(float(accuracy), 4),
    "macro_precision": round(float(precision), 4),
    "macro_recall": round(float(recall), 4),
    "macro_f1_score": round(float(f1), 4),
    "target_source": target_column,
    "note": (
        "Prototype evaluation. Use human_verified_label for strict final evaluation "
        "when manually verified labels are available."
    )
}

with open(RESULTS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4)

print("Saved model and evaluation files.")

## 7. MLP-based ensemble inference

In [ ]:
def assign_status(item):
    if item["agreement_count"] >= 2 and item["meta_confidence"] >= 0.70:
        return "High Confidence"
    if item["agreement_count"] == 1 and item["meta_confidence"] >= 0.90:
        return "Medium Confidence"
    return "Needs Review"


def meta_predict_group(group):
    feature_row = build_meta_feature_row(
        group=group,
        image_name="inference",
        provisional_label=choose_provisional_label(group)
    )

    X_group = pd.DataFrame([feature_row]).reindex(columns=feature_columns, fill_value=0)
    predicted_id = meta_classifier.predict(X_group)[0]
    predicted_label = label_encoder.inverse_transform([predicted_id])[0]
    probabilities = meta_classifier.predict_proba(X_group)[0]

    best_det = max(group, key=lambda d: d["confidence"])
    item = {
        "label": predicted_label,
        "meta_confidence": float(max(probabilities)),
        "best_yolo_confidence": float(best_det["confidence"]),
        "box": best_det["box"],
        "agreement_count": len(set(d["model"] for d in group)),
        "agreement_models": sorted(set(d["model"] for d in group)),
        "merged_labels": sorted(set(d["label"] for d in group)),
        "raw_group_size": len(group)
    }
    item["status"] = assign_status(item)
    return item


META_OUTPUT_DIR = BACKEND_DIR / "meta_ensemble_outputs"
if META_OUTPUT_DIR.exists():
    shutil.rmtree(META_OUTPUT_DIR)
META_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_meta_records = []

for image_path in test_images:
    detections = []

    for member_name, model in models.items():
        result = model(str(image_path), conf=0.25, verbose=False)[0]
        mapping = {int(k): v for k, v in global_label_mapping[member_name].items()}
        detections.extend(extract_detections(result, mapping, member_name))

    results = [
        meta_predict_group(group)
        for group in group_related_detections(detections, iou_threshold=0.30)
    ]

    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    for item in results:
        x1, y1, x2, y2 = item["box"]
        text = (
            f"{item['label']} | MLP={item['meta_confidence']:.2f} | "
            f"{item['agreement_count']} model | {item['status']}"
        )
        draw.rectangle([x1, y1, x2, y2], outline="red", width=4)
        draw.text((x1, max(0, y1 - 20)), text, fill="red")

        all_meta_records.append({
            "image": image_path.name,
            "final_label": item["label"],
            "meta_confidence": round(item["meta_confidence"], 3),
            "best_yolo_confidence": round(item["best_yolo_confidence"], 3),
            "agreement_count": item["agreement_count"],
            "agreement_models": ", ".join(item["agreement_models"]),
            "merged_labels": ", ".join(item["merged_labels"]),
            "raw_group_size": item["raw_group_size"],
            "status": item["status"]
        })

    image.save(META_OUTPUT_DIR / f"meta_{image_path.stem}.jpg")

meta_results_csv = BACKEND_DIR / "meta_ensemble_results.csv"
pd.DataFrame(all_meta_records).to_csv(meta_results_csv, index=False)

print("Saved annotated images:", META_OUTPUT_DIR)
print("Saved CSV:", meta_results_csv)
print("Total meta detections:", len(all_meta_records))

## 8. Export notebook copies to the submission folder

This notebook is the combined version. The final cell copies it into the required submission folder.  
For the teacher's requested structure, you can keep this as `meta_classifier_full_pipeline.ipynb` and optionally duplicate it as `feature_extraction.ipynb` and `meta_classifier_training.ipynb`.

In [ ]:
# Run this only after uploading/copying this notebook into Colab or Drive.
# It is kept here as a reminder for final submission preparation.
print("Full pipeline completed.")